# §1.9.3 — 훈련 구간 밖에서 무슨 일이 일어나는가

> 딥러닝 교재 · 1부 1장 9절 3항 (🐍)
> 선행: §1.7.1(콤팩트 조건) · §1.8.3(바깥 기울기) · §1.9.2(볼록 껍질 밖에서 아핀)

## 이 노트북이 답하는 질문

1. **§1.9.2의 정리가 맞는가?** 학습된 망이 자료 밖에서 정말 아핀인지 수치로 검산한다.
2. **다른 모형족은 밖에서 무엇을 하는가?** 신경망 · 다항식 · 가우시안 과정을 나란히 놓는다.
3. **모형이 "여기는 모른다"고 말할 수 있는가?** 셋 중 하나만 말한다 — 그리고 그것도 조건부다.

**예상 실행 시간** CPU 단일 코어 약 40초 (`FAST = True`이면 약 15초).

이 노트북의 결론은 "어느 모형이 외삽을 잘하는가"가 **아니다.** 자료가 없는 곳의 거동은 자료가 정하지 않으므로,
**모형족의 가정이 정한다.** 물어야 할 것은 "이 모형은 밖에서 무엇을 가정하는가"이다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260804
N_TRAIN  = 40        # 훈련점 수
NOISE    = 0.05      # 관측 잡음 표준편차
X_TRAIN  = 1.0       # 훈련 구간 [-X_TRAIN, X_TRAIN]
X_VIEW   = 3.0       # 그림 구간 [-X_VIEW, X_VIEW]
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_1_9_3_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 문제 설정

참 함수는 **아핀으로도 다항식으로도 자연스럽게 뻗지 않는 것**을 고른다.

$$g(x) = \sin(2x) + 0.3\,x$$

주기 성분과 선형 추세가 섞여 있으므로 세 모형족의 가정이 모두 어긋난다.
훈련은 $[-1, 1]$ 에서만 하고 예측은 $[-3, 3]$ 에서 본다.
1차원이므로 볼록 껍질은 $[\min_i x_i, \max_i x_i]$ 이다.

In [ ]:
g = lambda x: np.sin(2*np.asarray(x, float)) + 0.3*np.asarray(x, float)

rng = np.random.default_rng(SEED)
x_tr = np.sort(rng.uniform(-X_TRAIN, X_TRAIN, N_TRAIN))
y_tr = g(x_tr) + rng.normal(0, NOISE, N_TRAIN)
HULL = (float(x_tr.min()), float(x_tr.max()))
x_q  = np.linspace(-X_VIEW, X_VIEW, 1201)

print(f"훈련점 {N_TRAIN}개, 볼록 껍질 = [{HULL[0]:.3f}, {HULL[1]:.3f}]")
print(f"예측 구간 [-{X_VIEW}, {X_VIEW}] 중 껍질 밖 비율: "
      f"{np.mean((x_q < HULL[0]) | (x_q > HULL[1])):.1%}")

### 1.1 세 모형족

| | 밖에서 무엇을 하리라 예상되는가 | 근거 |
|---|---|---|
| ReLU 망 | **아핀**으로 뻗는다 | §1.9.2 정리 |
| 다항식 (차수 $k$) | $x^{k}$ 로 **발산**한다 | 최고차항이 지배 |
| 가우시안 과정 (RBF) | **사전 평균으로 회귀**하고 분산이 커진다 | 핵이 거리에 따라 감쇠 |

**불확실성을 어떻게 얻는지가 셋 모두 다르다.** 가우시안 과정은 사후 분산이 폐형식으로 나오고,
다항식은 모형이 맞다는 가정 아래의 예측 분산이 있으며, **신경망은 없다** — 대용품을 따로 만들어야 한다(6절).

가우시안 과정은 8부에서 본격적으로 다룬다. 여기서는 쓸 두 식만 적는다.

$$\mu(x_*) = k_*^{\top}(K + \sigma_n^2 I)^{-1} y, \qquad
s^2(x_*) = k(x_*, x_*) - k_*^{\top}(K + \sigma_n^2 I)^{-1} k_*$$

읽는 법: **훈련점들과 얼마나 닮았는지로 예측하고, 닮은 점이 없으면 분산이 사전 분산으로 돌아간다.**

In [ ]:
# ── ReLU 완전연결망 (§1.8.3과 동일한 구현) ──
def init_net(L, w, rng, bstd=0.5):
    dims = [1] + [w]*L + [1]; Ws, bs = [], []
    for i in range(len(dims)-1):
        Ws.append(rng.normal(0, np.sqrt(2.0/dims[i]), (dims[i], dims[i+1])))
        bs.append(rng.normal(0, bstd, dims[i+1]) if i < len(dims)-2 else np.zeros(dims[i+1]))
    return Ws, bs

def forward(x, Ws, bs):
    a = np.asarray(x, float).reshape(-1, 1)
    for i in range(len(Ws)-1):
        a = np.maximum(a @ Ws[i] + bs[i], 0)
    return (a @ Ws[-1] + bs[-1]).ravel()

def train(Ws, bs, x, y, steps, lr=5e-3):
    mW=[np.zeros_like(W) for W in Ws]; vW=[np.zeros_like(W) for W in Ws]
    mb=[np.zeros_like(b) for b in bs]; vb=[np.zeros_like(b) for b in bs]
    n = len(x)
    for t in range(1, steps+1):
        a = x.reshape(-1,1); acts=[a]; pre=[]
        for i in range(len(Ws)-1):
            z = a @ Ws[i] + bs[i]; pre.append(z); a = np.maximum(z,0); acts.append(a)
        d = (a @ Ws[-1] + bs[-1]).ravel() - y
        gr = (2.0/n) * d.reshape(-1,1)
        gW=[None]*len(Ws); gb=[None]*len(bs)
        gW[-1] = acts[-1].T @ gr; gb[-1] = gr.sum(0); delta = gr @ Ws[-1].T
        for i in range(len(Ws)-2, -1, -1):
            delta = delta * (pre[i] > 0)
            gW[i] = acts[i].T @ delta; gb[i] = delta.sum(0)
            if i > 0:
                delta = delta @ Ws[i].T
        for i in range(len(Ws)):
            for (p, gp, m, v) in ((Ws[i], gW[i], mW, vW), (bs[i], gb[i], mb, vb)):
                m[i] = 0.9*m[i] + 0.1*gp; v[i] = 0.999*v[i] + 0.001*gp**2
                p -= lr*(m[i]/(1-0.9**t))/(np.sqrt(v[i]/(1-0.999**t)) + 1e-8)
    return float(np.mean(d**2))

# ── 가우시안 과정 ──
def gp_posterior(xt, yt, xq, kern, sn=NOISE):
    K  = kern(xt[:, None], xt[None, :]) + sn**2*np.eye(len(xt))
    Ks = kern(xq[:, None], xt[None, :])
    Kss = kern(xq[:, None], xq[:, None]).ravel()
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, yt))
    v = np.linalg.solve(L, Ks.T)
    return Ks @ alpha, np.sqrt(np.maximum(Kss - np.sum(v**2, axis=0), 1e-12))

k_rbf  = lambda a, b: np.exp(-(a-b)**2/(2*0.4**2))
k_per  = lambda a, b: np.exp(-2*np.sin(np.pi*np.abs(a-b)/np.pi)**2/1.0**2)
k_comp = lambda a, b: k_per(a, b) + 0.1*(1.0 + a*b)

STEPS = 2000 if FAST else 4000
Ws, bs = init_net(3, 64, np.random.default_rng(1))
mse_nn = train(Ws, bs, x_tr, y_tr, STEPS)
y_nn = forward(x_q, Ws, bs)

KPOLY = 9
Vt = np.vander(x_tr, KPOLY+1)
w_poly = np.linalg.lstsq(Vt, y_tr, rcond=None)[0]
y_poly = np.vander(x_q, KPOLY+1) @ w_poly

mu_gp, sd_gp = gp_posterior(x_tr, y_tr, x_q, k_rbf)

print(f"신경망 훈련 MSE = {mse_nn:.5f}")
print(f"다항 차수 {KPOLY} 훈련 MSE = {np.mean((Vt@w_poly - y_tr)**2):.5f}")
print(f"GP(RBF) 훈련점에서의 평균 표준편차 = "
      f"{gp_posterior(x_tr,y_tr,x_tr,k_rbf)[1].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4), sharey=True)
panels = [(y_nn, None, lab('ReLU 망', 'ReLU network'), CB[4]),
          (y_poly, None, lab(f'다항식 (차수 {KPOLY})', f'polynomial (deg {KPOLY})'), CB[5]),
          (mu_gp, sd_gp, lab('가우시안 과정 (RBF)', 'Gaussian process (RBF)'), CB[3])]
for ax, (m, s, ttl, c) in zip(axes, panels):
    ax.axvspan(HULL[0], HULL[1], color='0.85', zorder=0)
    ax.plot(x_q, g(x_q), color=CB[0], lw=1.2, ls='--', label=lab('참 함수', 'truth'))
    if s is not None:
        ax.fill_between(x_q, m-2*s, m+2*s, color=c, alpha=0.22,
                        label=lab('±2 표준편차', r'$\pm 2$ s.d.'))
    ax.plot(x_q, m, color=c, lw=1.8, label=lab('예측', 'prediction'))
    ax.plot(x_tr, y_tr, 'o', ms=3.5, color=CB[0], label=lab('훈련 자료', 'training data'))
    ax.set_ylim(-4, 4); ax.set_xlabel('$x$'); ax.set_title(ttl, fontsize=10)
    ax.legend(fontsize=7, loc='lower right')
axes[0].set_ylabel('$y$')
fig.suptitle(lab('회색 구간이 훈련 범위 — 밖에서는 셋이 전부 다른 방향으로 간다',
                 'grey band is the training range'), y=1.04, fontsize=10)
show('three_families')

> ### 읽을 것 셋
>
> **(가) 셋이 밖에서 전부 다른 방향으로 간다.** 그리고 자료는 세 방향 중 어느 것도 지지하지 않는다.
> 차이를 만든 것은 자료가 아니라 **편향**이다 — §1.4.4의 결론이 여기서 다시 나온다.
>
> **(나) 불확실성 띠가 있는 것은 하나뿐이다.** 신경망과 다항식은 밖에서도 확신에 찬 선을 그린다.
>
> **(다) 어느 것도 참 함수를 맞히지 못한다.** 가우시안 과정도 밖에서 **틀린다** — 다만 **틀렸다고 말한다.**
> 문제는 예측이 나쁘다는 것이 아니라, **좋을 근거가 없는데 확신한다는 것**이다.

---
## 2. §1.9.2의 정리를 수치로 검산

정리는 "가장 바깥 꺾임 너머에서 아핀"이라고 했다. 먼저 **꺾임이 실제로 어디에 있는지** 찾고,
그 바깥에서 이차 차분이 0인지 본다.

In [ ]:
x_wide = np.linspace(-20, 20, 200001 if FAST else 400001)
y_wide = forward(x_wide, Ws, bs)
sl = np.diff(y_wide)/np.diff(x_wide)
dd = np.abs(np.diff(sl)); sc = np.maximum(np.abs(sl[:-1]), np.abs(sl[1:]))
kink_x = x_wide[np.flatnonzero(dd > 1e-6*(1+sc)) + 1]
print(f"꺾임 위치 범위: [{kink_x.min():.3f}, {kink_x.max():.3f}]  (훈련 범위는 [{HULL[0]:.2f}, {HULL[1]:.2f}])")
print("-> 꺾임이 훈련 범위 밖에도 놓인다. 정리가 말하는 것은 '가장 바깥 꺾임 너머'다.\n")

for side, m in [(lab('왼쪽','left'), x_wide < kink_x.min()-0.01),
                (lab('오른쪽','right'), x_wide > kink_x.max()+0.01)]:
    yy = y_wide[m]; s2 = np.diff(yy)/np.diff(x_wide[m])
    d2 = np.abs(np.diff(yy, 2)).max()
    print(f"  {side} 바깥: 기울기 {s2.mean():+.6f}  (표준편차 {s2.std():.1e}),  이차차분 최대 {d2:.1e}")
    assert d2 < 1e-9, "아핀이 아니다"
print("\n§1.9.2 확인: 가장 바깥 꺾임 너머에서 이차 차분이 기계 정밀도로 0이다.")

In [ ]:
# 바깥 기울기는 무엇이 정하는가 — 시드를 바꿔 본다
print("같은 자료, 다른 초기화 시드에서의 오른쪽 바깥 기울기")
slopes = []
for s_ in range(3 if FAST else 6):
    W2, b2 = init_net(3, 64, np.random.default_rng(100+s_))
    train(W2, b2, x_tr, y_tr, STEPS)
    yy = forward(np.array([50.0, 51.0]), W2, b2)
    slopes.append(float(yy[1]-yy[0]))
    print(f"  seed {s_}: {slopes[-1]:+.4f}")
print(f"\n범위 [{min(slopes):+.3f}, {max(slopes):+.3f}]  "
      f"부호가 갈리는가: {'예' if min(slopes)*max(slopes) < 0 else '아니오'}")
print("-> §1.9.2 (나): 외삽 방향은 가장자리 몇 점이 정하며, 그 점들은 가장 신뢰할 수 없는 자료다.")

---
## 3. 볼록 껍질 안이라고 안전한가 — 내삽 구멍

훈련 구간 **가운데를 비운다.** 껍질 안이지만 자료가 없는 영역이 생긴다.
§1.9.2는 껍질 밖에 대한 정리였는데, 실제 위험은 **"자료가 없는 곳" 전반**이다.

In [ ]:
HOLE = (-0.35, 0.35)
mask = (x_tr < HOLE[0]) | (x_tr > HOLE[1])
xh, yh = x_tr[mask], y_tr[mask]

Wh, bh = init_net(3, 64, np.random.default_rng(1))
train(Wh, bh, xh, yh, STEPS)
y_nn_h = forward(x_q, Wh, bh)
mu_h, sd_h = gp_posterior(xh, yh, x_q, k_rbf)

inh = (x_q > HOLE[0]) & (x_q < HOLE[1])
print(f"구멍 {HOLE}, 남은 훈련점 {len(xh)}개")
print(f"  GP  구멍 안 평균 표준편차 {sd_h[inh].mean():.4f}  vs 자료 있는 곳 "
      f"{sd_h[(x_q>0.6)&(x_q<1.0)].mean():.4f}")

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.axvspan(*HOLE, color=CB[1], alpha=0.18, zorder=0,
           label=lab('자료를 비운 구간', 'removed region'))
ax.plot(x_q, g(x_q), color=CB[0], lw=1.2, ls='--', label=lab('참 함수','truth'))
ax.fill_between(x_q, mu_h-2*sd_h, mu_h+2*sd_h, color=CB[3], alpha=0.22)
ax.plot(x_q, mu_h, color=CB[3], lw=1.6, label=lab('GP (RBF)','GP (RBF)'))
ax.plot(x_q, y_nn_h, color=CB[4], lw=1.6, label=lab('ReLU 망','ReLU net'))
ax.plot(xh, yh, 'o', ms=3.5, color=CB[0])
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-2.2, 2.2)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(lab('껍질 안이어도 자료가 없으면 근거가 없다',
                 'inside the hull but without data'), fontsize=10)
ax.legend(fontsize=8)
show('interpolation_hole')

> 구멍 안에서 가우시안 과정의 띠는 부풀지만 **신경망은 아무 일 없이 매끄럽게 지나간다.**
> §1.9.2의 정리는 껍질 밖에 대한 것이었으나, **위험은 "자료가 없는 곳" 전반**이다.

---
## 4. ⚠︎ 불확실성이 있다고 안전한 것은 아니다

가우시안 과정의 띠는 **핵이 맞다는 가정** 아래에서의 불확실성이다. 핵을 바꿔 본다.

In [ ]:
kerns = [('RBF', k_rbf, CB[3]),
         (lab('주기 핵','periodic'), k_per, CB[5]),
         (lab('주기 + 선형','periodic + linear'), k_comp, CB[6])]
fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.3), sharey=True)
print("x = -3 에서 (참값 %.3f)" % g(-3))
for ax, (nm, k, c) in zip(axes, kerns):
    m, s = gp_posterior(x_tr, y_tr, x_q, k)
    ax.axvspan(HULL[0], HULL[1], color='0.85', zorder=0)
    ax.plot(x_q, g(x_q), color=CB[0], lw=1.2, ls='--')
    ax.fill_between(x_q, m-2*s, m+2*s, color=c, alpha=0.22)
    ax.plot(x_q, m, color=c, lw=1.8)
    ax.plot(x_tr, y_tr, 'o', ms=3, color=CB[0])
    ax.set_ylim(-4, 4); ax.set_xlabel('$x$'); ax.set_title(nm, fontsize=10)
    err = abs(m[0] - g(-3))
    print(f"  {nm:>18}: 예측 {m[0]:+.3f}  표준편차 {s[0]:.3f}  오차 {err:.3f}")
axes[0].set_ylabel('$y$')
fig.suptitle(lab('핵도 편향이다 — 주기 핵은 좁은 띠로 틀린 답을 확신한다',
                 'the kernel is a bias too'), y=1.04, fontsize=10)
show('kernel_is_a_bias')

> **주기 핵을 보십시오.** 띠가 좁은데 답이 틀렸습니다. **확신에 차서 틀린 것**이며,
> 아무 띠도 없는 신경망보다 나은지 분명하지 않습니다.
>
> §1.5의 논지가 다시 적용된다 — 핵도 귀납 편향이고, 맞을 때만 도움이 된다.
> 가정을 훨씬 약하게 하는 대안이 §50.7(적합성 예측)이며, §50.7.7이 그것도 교환 가능성이 깨지면 무너진다고 경고한다.

---
## 5. 신경망 앙상블은 대안이 되는가

서로 다른 초기화로 학습한 망 여럿의 예측 산포를 불확실성 대용으로 쓴다.

In [ ]:
NENS = 5 if FAST else 10
preds = []
for s_ in range(NENS):
    W3, b3 = init_net(3, 64, np.random.default_rng(200+s_))
    train(W3, b3, x_tr, y_tr, STEPS)
    preds.append(forward(x_q, W3, b3))
P = np.array(preds); mu_e, sd_e = P.mean(0), P.std(0)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.axvspan(HULL[0], HULL[1], color='0.85', zorder=0)
for p in P:
    ax.plot(x_q, p, color=CB[4], lw=0.7, alpha=0.45)
ax.plot(x_q, g(x_q), color=CB[0], lw=1.3, ls='--', label=lab('참 함수','truth'))
ax.plot(x_q, mu_e, color=CB[4], lw=1.8, label=lab(f'앙상블 평균 ({NENS}개)', f'ensemble mean ({NENS})'))
ax.plot(x_tr, y_tr, 'o', ms=3.5, color=CB[0])
ax.set_ylim(-4, 4); ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(lab('앙상블 산포는 커지지만 체계적이지 않다',
                 'ensemble spread grows but is not systematic'), fontsize=10)
ax.legend(fontsize=8)
show('nn_ensemble')

inside = np.abs(x_q) < HULL[1]
print(f"앙상블 표준편차: 훈련 구간 안 {sd_e[inside].mean():.4f} · "
      f"|x|>2 {sd_e[np.abs(x_q)>2].mean():.4f}")
print(f"GP(RBF) 표준편차: 훈련 구간 안 {sd_gp[inside].mean():.4f} · "
      f"|x|>2 {sd_gp[np.abs(x_q)>2].mean():.4f}")
print("\n-> 앙상블도 밖에서 벌어지기는 한다. 다만 모든 망이 같은 방향으로 뻗으면")
print("   산포가 작게 나올 수 있고, 그러면 확신에 차 있으면서 동시에 틀린다. §50.3에서 다룬다.")

---
## 6. 거리에 따른 오차

In [ ]:
print("  |x|      신경망       다항 %d        GP(RBF)     GP 표준편차" % KPOLY)
for t in [1.0, 1.5, 2.0, 2.5, 3.0]:
    i = int(np.argmin(np.abs(x_q - t))); tr = g(x_q[i])
    print(f"  {t:.1f}   {abs(y_nn[i]-tr):9.3f}  {abs(y_poly[i]-tr):12.1f}  "
          f"{abs(mu_gp[i]-tr):9.3f}   {sd_gp[i]:9.3f}")
print("\n-> 다항식이 가장 빨리 무너진다. GP는 오차가 작아 보이지만 그것은")
print("   사전 평균 0이 우연히 참값과 가까웠기 때문이며, 표준편차가 그것을 정직하게 말한다.")

---
## 7. 자기 점검

1. 2절에서 꺾임이 훈련 범위 **밖**에도 놓였다. 왜인가? 이것이 §1.9.2의 정리와 모순되는가?
2. `X_TRAIN` 을 2.0으로 늘리면 세 그림이 어떻게 달라지는가? **자료를 늘린 것과 범위를 넓힌 것 중 무엇이 효과가 있는가?**
3. 4절에서 주기 핵의 띠가 좁았다. 그 띠가 **정직하려면** 무엇이 참이어야 하는가?
4. §1.8.3에서 깊은 망의 바깥 기울기가 $2^L$까지 커졌다. 이 노트북의 망을 깊이 8로 바꾸면 외삽이 어떻게 되겠는가?

In [ ]:
# 자기 점검 2의 확인 — 자료 '수'를 늘리는 것과 '범위'를 넓히는 것
OUT = np.abs(x_q) > 2.0
NSD = 2 if FAST else 3

def outer_err(n_, xr, tag):
    e = []
    for s_ in range(NSD):
        r = np.random.default_rng(1000 + s_)
        xt = np.sort(r.uniform(-xr, xr, n_)); yt = g(xt) + r.normal(0, NOISE, n_)
        W4, b4 = init_net(3, 64, np.random.default_rng(s_))
        train(W4, b4, xt, yt, STEPS//2)
        e.append(np.mean(np.abs(forward(x_q, W4, b4)[OUT] - g(x_q[OUT]))))
    print(f"  {tag}: |x|>2 평균 절대오차 {np.mean(e):.3f}  (시드 표준편차 {np.std(e):.3f})")

print(f"범위 ±1 고정, 자료 수만 늘리기 (시드 {NSD}개 평균, 학습 {STEPS//2} step)")
for n_ in ([40, 160, 640] if FAST else [40, 160, 640, 2560]):
    outer_err(n_, 1.0, f"n={n_:>5}")
print("\n자료 수 160 고정, 범위만 넓히기")
for xr in [1.0, 2.0, 3.0]:
    outer_err(160, xr, f"범위 ±{xr}")
print("\n-> 같은 범위에서 자료를 60배 늘려도 바깥 오차는 포화한다.")
print("   범위를 넓히면 곧바로 사라진다. 외삽은 자료량 문제가 아니라 구조 문제다 (§1.4.2).")

---
## 8. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `X_TRAIN` | 0절 | 1.0 | 훈련 범위. **바깥 거동을 바꾸는 유일한 손잡이** |
| `N_TRAIN` | 0절 | 40 | 훈련점 수. 늘려도 바깥은 거의 그대로 |
| `NOISE` | 0절 | 0.05 | 잡음. 크면 가장자리 기울기가 더 요동친다 |
| `KPOLY` | 1절 | 9 | 다항 차수. 낮추면 발산이 완만해지지만 안쪽이 나빠진다 |
| 망의 깊이·폭 | 1절 | 3, 64 | 깊게 하면 바깥 기울기가 가팔라진다 (§1.8.3) |
| `HOLE` | 3절 | (−0.35, 0.35) | 내삽 구멍의 크기 |
| `k_rbf` 의 길이 척도 | 1절 | 0.4 | 크게 하면 GP가 더 멀리까지 확신한다 |

**권하는 첫 실험** — `k_rbf` 의 길이 척도를 0.4에서 3.0으로 바꾸십시오. 가우시안 과정의 띠가
훈련 범위 밖에서도 **좁게 유지**됩니다. "모른다"고 말하는 능력조차 **길이 척도라는 손잡이가 정한 것**이며,
그 손잡이는 자료가 아니라 사람이 골랐습니다 (§1.3.4).

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")